# Football Player Positioning AI - LSTM Training

This notebook trains per-player LSTM models on Google Colab with free GPU.

**Steps:**
1. Clone repo & install dependencies
2. Download Metrica Sports data
3. Run preprocessing pipeline
4. Run feature engineering
5. Train LSTM models for all players
6. Download trained models

## 0. Check GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU')

## 1. Clone Repository

In [ ]:
!git clone https://github.com/Sysus611/football-positioning-ai.git
%cd football-positioning-ai
!pip install pandas numpy pyarrow scipy pyyaml matplotlib -q

## 2. Download Metrica Sports Data

In [ ]:
import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/data"

FILES = {
    'game1': ['tracking_home.csv', 'tracking_away.csv'],
    'game2': ['tracking_home.csv', 'tracking_away.csv'],
    'game3': ['tracking.txt', 'metadata.xml', 'metadata.json'],
}

# Map game dirs to URL paths
URL_MAP = {'game1': 'Sample_Game_1', 'game2': 'Sample_Game_2', 'game3': 'Sample_Game_3'}

for game, files in FILES.items():
    game_dir = f'data/raw/metrica/{game}'
    os.makedirs(game_dir, exist_ok=True)
    url_game = URL_MAP[game]
    for fname in files:
        url = f'{BASE_URL}/{url_game}/{fname}'
        dest = f'{game_dir}/{fname}'
        if os.path.exists(dest):
            print(f'  Already exists: {dest}')
            continue
        print(f'  Downloading {url_game}/{fname}...', end='')
        urllib.request.urlretrieve(url, dest)
        size_mb = os.path.getsize(dest) / (1024*1024)
        print(f' OK ({size_mb:.1f} MB)')

print('Download complete!')

## 3. Run Preprocessing

In [ ]:
!python src/preprocessing/preprocess.py

## 4. Feature Engineering

In [ ]:
!python src/features/build_features.py

## 5. Train All Players

This will train one LSTM model per player. With T4 GPU, each player takes ~5-10 minutes.

In [ ]:
!python src/training/train.py

## 6. Check Results & Download Models

In [ ]:
import os
import torch

model_dir = 'data/models'
if os.path.exists(model_dir):
    models = sorted(os.listdir(model_dir))
    print(f'Trained {len(models)} models:\n')
    for m in models:
        path = os.path.join(model_dir, m)
        ckpt = torch.load(path, map_location='cpu')
        print(f"  {m:20s} | best_val_loss: {ckpt['best_val_loss']:.6f} | epoch: {ckpt['epoch']}")
else:
    print('No models found!')

In [ ]:
# Zip models for download
!zip -r /content/trained_models.zip data/models/

from google.colab import files
files.download('/content/trained_models.zip')
print('Download started! Check your browser downloads.')

## 7. Quick Visualization: Predict vs Actual

Pick a player and visualize prediction vs ground truth on a random sample.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')
from src.model.lstm_baseline import PlayerLSTM

# Pick a player (change this to test different players)
PLAYER_ID = 'home_11'

# Load model
ckpt = torch.load(f'data/models/{PLAYER_ID}.pt', map_location='cpu')
model = PlayerLSTM(input_dim=ckpt['input_dim'], pred_frames=ckpt['pred_frames'])
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Load test data
data = np.load(f'data/tensors/{PLAYER_ID}.npz')
X, Y = data['X'], data['Y']

# Pick 4 random samples
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'{PLAYER_ID} - Predicted vs Actual Trajectory', fontsize=14)

for ax_idx, ax in enumerate(axes.flat):
    idx = np.random.randint(len(X))
    x_input = torch.from_numpy(X[idx:idx+1])

    with torch.no_grad():
        pred = model(x_input).numpy()[0]  # [pred_frames, 2]

    actual = Y[idx]  # [pred_frames, 2]

    # Also show the observation trajectory (target player position is features[-4:-2])
    obs_x = X[idx, :, -4]  # target player x during observation
    obs_y = X[idx, :, -3]  # target player y during observation

    # Draw pitch outline
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(1.05, -0.05)  # Flip y
    ax.set_facecolor('#2d5a27')
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor='white', linewidth=1))
    ax.axvline(x=0.5, color='white', linewidth=0.5, alpha=0.5)

    # Plot trajectories
    ax.plot(obs_x, obs_y, 'c-', linewidth=1.5, alpha=0.6, label='Observation (3s)')
    ax.plot(actual[:, 0], actual[:, 1], 'w-', linewidth=2, label='Actual')
    ax.plot(pred[:, 0], pred[:, 1], 'r--', linewidth=2, label='Predicted')

    # Start/end points
    ax.scatter(obs_x[0], obs_y[0], c='cyan', s=50, zorder=5, marker='s')
    ax.scatter(actual[-1, 0], actual[-1, 1], c='white', s=80, zorder=5, marker='*')
    ax.scatter(pred[-1, 0], pred[-1, 1], c='red', s=80, zorder=5, marker='*')

    ax.set_title(f'Sample #{idx}', color='white')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('prediction_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: prediction_visualization.png')